# 🏠 Housing Price Prediction with Robbins-Monro Algorithm

Welcome! In this notebook, you will learn how to implement the **Robbins-Monro stochastic approximation algorithm** to predict house prices.

## What You Will Learn

We will progress through the following stages:

| Stage | Topic | What You'll Do |
|-------|-------|----------------|
| 1 | Understanding the Data | Load and explore a real housing dataset |
| 2 | Single House (Online) | Predict price using ONE house at a time |
| 3 | Multiple Houses (Batch) | Use ALL houses together for better learning |
| 4 | Multiple Features | Use BOTH area AND age to predict price |

---

## ⚠️ Important Notes for Beginners

- **Don't worry if code looks confusing!** We'll explain every line.
- **Run cells in order** - click a cell and press `Shift + Enter\Command` to run it.
- **Solutions are built in an iterative fashion** - after you implement a function see how you can reuse it (call it) in order to complete the next task!
- **Read the comments** - they explain what each line does.
- **Ask questions!** - If something is unclear, that's normal!

---

## 📦 Step 0: Install Required Packages

First, we need to install some tools (called "packages" or "libraries") that help us work with data and create visualizations.

**Run this cell once** - it may take a minute to install everything.

In [ ]:
#@title ▶️ Run this cell to install packages (click to expand code)
!pip install kagglehub pandas matplotlib plotly ipywidgets --quiet
print("✅ All packages installed successfully!")

## 📦 Step 1: Import Libraries

Now we need to "import" these packages into our notebook so we can use them.

Think of it like this:
- Installing = Buying a toolbox and putting it in your garage
- Importing = Taking the toolbox out so you can use the tools

In [ ]:
#@title ▶️ Run this cell to import libraries (click to expand code)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import kagglehub
import plotly.express as px
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider, IntSlider
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

print("✅ All libraries imported successfully!")
print("")
print("📚 Libraries we're using:")
print("   • pandas    → Working with data tables")
print("   • matplotlib → Creating static charts")
print("   • numpy     → Mathematical operations")
print("   • kagglehub → Downloading datasets")
print("   • plotly    → Creating interactive charts")
print("   • ipywidgets → Creating sliders and buttons")

---

# 📊 Stage 1: Understanding the Dataset

## What is a Dataset?

A dataset is just a collection of information organized in rows and columns (like an Excel spreadsheet).

## Our Dataset

We'll use a synthetic (computer-generated) house price dataset from Kaggle containing **10,000 houses**.

Each house has these features:
- **square_feet**: Size of the house (in square feet)
- **num_rooms**: Number of rooms
- **age**: Age of the house (in years)
- **distance_to_city(km)**: Distance from city center (in km)
- **price**: The house price (this is what we want to predict!)

## Automatic Download

The code below will **automatically download** the dataset from Kaggle. No manual download needed!

In [ ]:
#@title ▶️ Run this cell to download dataset (click to expand code)
print("📥 Downloading dataset from Kaggle...")
print("   (This may take a moment the first time)")
print("")

dataset_path = kagglehub.dataset_download('muhamedumarjamil/house-price-prediction-dataset')

print(f"✅ Dataset downloaded!")
print(f"📁 Saved to: {dataset_path}")

In [ ]:
#@title ▶️ Run this cell to load dataset (click to expand code)
import os

csv_file = os.path.join(dataset_path, 'house_prices_dataset.csv')
df = pd.read_csv(csv_file)

print("✅ Dataset loaded into Python!")
print(f"")
print(f"📊 The dataset has:")
print(f"   • {len(df)} rows (houses)")
print(f"   • {len(df.columns)} columns (features)")

In [ ]:
#@title ▶️ Run this cell to see the first 5 houses (click to expand code)
print("🏠 First 5 houses in our dataset:")
print("")
df.head()

### 📌 Data Processing

Now we need to **extract** the specific columns we'll use:

- For **Stages 2-3**: We only need `size` (area) and `price`
- For **Stage 4**: We also need `age`

We'll convert these columns into **NumPy arrays** (lists of numbers) for easier math.

In [ ]:
#@title ▶️ Run this cell to extract data columns (click to expand code)
areas = df['square_feet'].values
prices = df['price'].values
ages = df['age'].values

print("✅ Data extracted successfully!")
print("")
print(f"📊 What we have:")
print(f"   • areas:  {len(areas)} values (house sizes in sq ft)")
print(f"   • ages:   {len(ages)} values (house ages in years)")
print(f"   • prices: {len(prices)} values (house prices in $)")
print("")
print(f"📋 First 5 examples:")
print(f"   Areas:  {areas[:5]}")
print(f"   Ages:   {ages[:5]}")
print(f"   Prices: {prices[:5]}")

In [ ]:
#@title ▶️ Run this cell to visualize the data (click to expand code)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(areas[:500], prices[:500], alpha=0.5, color='blue')
plt.xlabel('Size (sq ft)')
plt.ylabel('Price ($)')
plt.title('House Size vs Price')

plt.subplot(1, 2, 2)
plt.scatter(ages[:500], prices[:500], alpha=0.5, color='green')
plt.xlabel('Age (years)')
plt.ylabel('Price ($)')
plt.title('House Age vs Price')

plt.tight_layout()
plt.show()

print("")
print("💡 What do you notice?")
print("   • Size vs Price: As size INCREASES, price INCREASES (positive relationship)")
print("   • Age vs Price: As age INCREASES, price... (what do you see?)")

---

# 🎯 Stage 2: Single House Prediction (Online Learning)

## The Goal

We want to predict a house's price based on its size.

## The Model

Our prediction formula is very simple:

$$\hat{y} = w \times x$$

Where:
- $x$ = area of the house (input - what we know)
- $w$ = weight (the number we need to find!)
- $\hat{y}$ = predicted price (output - what we want to know)

**Example**: If $w = 200$ and house size is 1500 sq ft:
- Predicted price = $200 \times 1500 = \$300,000$

## The Problem

We don't know what $w$ should be! We need to **learn** it from data.

## The Algorithm (Fixed-Step)

Here's our simple learning algorithm:

```
1. Start with a random guess for w
2. Make a prediction: predicted_price = w × size
3. Calculate the error: how wrong were we?
   error = actual_price - predicted_price
4. Update w based on the error:
   - If error > 0 (we predicted too LOW): increase w
   - If error < 0 (we predicted too HIGH): decrease w
5. Repeat many times!
```

### 📝 Task 2.1: Implement the Prediction Function

**Your task**: Complete the function below that predicts a house price.

**What to do**: 
- The function receives `x` (area) and `w` (weight)
- Return the predicted price: `w * x`

In [ ]:
# ============================================
# TASK 2.1: YOUR CODE HERE
# ============================================
# Complete the function below

def predict(x, w):
    """
    Predict the house price.
    
    What this function does:
        Takes a house size and a weight, returns the predicted price.
    
    Parameters (inputs):
        x: The area of the house (in square feet)
           Example: 1500
        
        w: The weight (price per square foot)
           Example: 200
    """
    
    # TODO: Replace 'pass' with your code
    pass

In [ ]:
#@title ▶️ Run this cell to test your predict function (click to expand code)
test_result = predict(1000, 200)

print("🧪 Testing the predict function:")
print(f"   Input: x = 1000 sq ft, w = $200 per sq ft")
print(f"   Expected: $200,000")
print(f"   Got: ${test_result}")

if test_result == 200000:
    print("   ✅ Correct!")
else:
    print("   ❌ Something's wrong...")

### 📝 Task 2.2: Implement the Loss Function

The **loss** (also called **error**) tells us how far off our prediction is.

**Formula**: 
$$\text{loss} = \text{actual price} - \text{predicted price}$$

**What the loss tells us**:
- **Positive loss** (loss > 0): We predicted TOO LOW (need to increase w)
- **Negative loss** (loss < 0): We predicted TOO HIGH (need to decrease w)
- **Zero loss** (loss = 0): Perfect prediction!

**Your task**: Complete the function to compute the loss.

In [ ]:
# ============================================
# TASK 2.2: YOUR CODE HERE
# ============================================

def compute_loss(x, y, w):
    """
    Compute the loss (error) for a single house.
    
    What this function does:
        Calculates how wrong our prediction is.
    
    Parameters (inputs):
        x: The area of the house (in square feet)
           Example: 1500
        
        y: The ACTUAL price of the house (what it really costs)
           Example: 280000
        
        w: The current weight we're using
           Example: 200
    """
    
    # TODO: Step 1 - Calculate the prediction using the predict() function
    
    # TODO: Step 2 - Calculate and return the loss
    
    pass

In [ ]:
#@title ▶️ Run this cell to test your compute_loss function (click to expand code)
test_loss = compute_loss(1000, 200000, 180)

print("🧪 Testing the compute_loss function:")
print(f"   House: 1000 sq ft")
print(f"   Actual price: $200,000")
print(f"   Our weight (w): $180 per sq ft")
print(f"   Our prediction: ${180 * 1000}")
print(f"   Loss: ${test_loss}")

if test_loss == 20000:
    print("   ✅ Correct! Positive loss means we predicted too LOW.")
else:
    print("   ❌ Something's wrong...")

### 📝 Task 2.3: Implement the Fixed-Step Learning Algorithm

Now let's put it all together! We'll create a **learning loop** that:

1. Calculates the loss
2. Updates w based on whether the loss is positive or negative
3. Repeats many times

**The update rule**:
```python
if loss > 0:     # We predicted too LOW
    w = w + alpha  # Increase w
elif loss < 0:   # We predicted too HIGH
    w = w - alpha  # Decrease w
```

Where `alpha` (α) is the **step size** - how much we change w each time.

In [ ]:
# ============================================
# TASK 2.3: YOUR CODE HERE
# ============================================
# Complete the training loop below

# --- Setup: Pick one house from our dataset ---
X = areas[0]    # X = the area of the first house in our dataset
Y = prices[0]   # Y = the actual price of the first house

print("🏠 Training on a single house:")
print(f"   Area: {X} sq ft")
print(f"   Actual Price: ${Y}")
print("")

# --- Initialize our parameters ---
w = 50           # Our initial guess for the weight (probably wrong!)
alpha = 1.0      # Step size: how much we change w each iteration
iterations = 100 # How many times we'll update w

# --- Track history for visualization ---
w_history = [w]  # Start with our initial guess

# --- The Learning Loop ---
for i in range(iterations):
    
    # TODO: Step 1 - Compute the loss
    
    # TODO: Step 2 - Update w based on the loss
    # If L > 0: we predicted too low
    # If L < 0: we predicted too high
    
    pass  # Remove this line and add your code
    
    # Save the new w value to our history
    w_history.append(w)

# --- Print Results ---
print(f"✅ Training complete!")
print(f"")
print(f"📊 Results:")
print(f"   Final w = {w}")
print(f"   Final prediction: ${predict(X, w):.2f}")
print(f"   Actual price: ${Y}")
print(f"   Error: ${abs(Y - predict(X, w)):.2f}")

In [ ]:
#@title ▶️ Run this cell to visualize the learning process (click to expand code)
plt.figure(figsize=(10, 5))

plt.plot(w_history, 'b-', linewidth=2, label='Our w value')

true_w = Y / X
plt.axhline(y=true_w, color='green', linestyle='--', linewidth=2, 
            label=f'Target w = {true_w:.2f}')

plt.xlabel('Iteration')
plt.ylabel('Weight (w)')
plt.title('Fixed-Step Learning: How w Changes Over Time')
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()

print("")
print("💡 What do you notice?")
print("   → The algorithm OSCILLATES around the target!")
print("   → It can't settle exactly on the target value.")
print("   → This is because fixed steps are too coarse when we're close.")

---

# 🚀 Stage 3: Multiple Houses (Batch Learning)

## The Problem with Single House Learning

In Stage 2, we learned from just ONE house. But what if that house was unusual?

**Solution**: Use MANY houses at once!

## Batch Learning

Instead of looking at one house:
1. Look at ALL houses
2. Calculate the loss for EACH house
3. Add up all the losses (total loss)
4. Update w based on the total loss

## Robbins-Monro Update (Proportional Updates)

Instead of fixed steps, we'll use **proportional updates**:

$$w_{new} = w_{old} + \alpha \times L$$

**Why is this better?**
- If the error is LARGE, we take a LARGE step
- If the error is SMALL, we take a SMALL step
- This helps us converge smoothly!

### 📝 Task 3.1: Implement Batch Loss Computation

**Your task**: Create a function that computes the TOTAL loss over many houses.

**What to do**:
1. Loop through each house
2. Compute the loss for that house
3. Add it to a running total
4. Return the total

In [ ]:
# ============================================
# TASK 3.1: YOUR CODE HERE
# ============================================

def compute_batch_loss(X_list, Y_list, w):
    """
    Compute the TOTAL loss over multiple houses.
    
    What this function does:
        Loops through all houses, adds up all their losses.
    
    Parameters (inputs):
        X_list: A list of house areas
                Example: [1000, 1500, 2000]
        
        Y_list: A list of actual house prices (same order as X_list)
                Example: [200000, 300000, 400000]
        
        w:      The current weight we're using
    
    Returns (output):
        The TOTAL loss (sum of all individual losses)
    
    Example:
        If we have 3 houses with losses: +100, -50, +30
        Total loss = 100 + (-50) + 30 = 80
    """
    
    # Start with zero total loss
    total_loss = 0
    
    # TODO: Loop through each house
    # len(X_list) gives us how many houses there are
    # range(len(X_list)) gives us: 0, 1, 2, ..., n-1
    
    pass  # Remove this and add your code
    
    return total_loss

In [ ]:
#@title ▶️ Run this cell to test your compute_batch_loss function (click to expand code)
X_sample = areas[:4]
Y_sample = prices[:4]

test_loss = compute_batch_loss(X_sample, Y_sample, 100)

print("🧪 Testing compute_batch_loss:")
print(f"   Using {len(X_sample)} houses")
print(f"   With w = 100")
print(f"   Total loss = {test_loss:.2f}")
if test_loss is not None:
    print("   ✅ Function works!")
else:
    print("   ❌ Function returned None - did you forget to remove 'pass'?")

### 📝 Task 3.2: Implement Batch Learning with Robbins-Monro

Now we'll use the **Robbins-Monro** update rule:

$$w = w + \alpha \times L$$

Where:
- $L$ = total loss (from all houses)
- $\alpha$ = learning rate (a small number like 0.0000001)

**Why so small α?** Because we're summing losses from many houses, the total is very large!

In [ ]:
# ============================================
# TASK 3.2: YOUR CODE HERE
# ============================================

# --- Setup: Use first 100 houses ---
X_train = areas[:100]   # First 100 areas
Y_train = prices[:100]  # First 100 prices

print(f"🏘️ Training on {len(X_train)} houses")
print("")

# --- Initialize parameters ---
w = 50                   # Initial guess
alpha = 0.0000001        # Learning rate (very small!)
epochs = 50              # Number of training rounds (epochs)

# --- Track history ---
w_history = [w]
loss_history = []

# --- The Training Loop ---
for epoch in range(epochs):
    
    # TODO: Step 1 - Compute the total loss over all houses
    # Hint: L = compute_batch_loss(X_train, Y_train, w)
    
    # TODO: Step 2 - Update w using Robbins-Monro rule
    # The update is PROPORTIONAL to the loss
    # Hint: w = w + alpha * L
    
    pass  # Remove this and add your code
    
    # Save history
    w_history.append(w)
    loss_history.append(abs(L))  # abs() = absolute value (no negatives)
    
    # Print progress every 10 epochs
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: w = {w:.4f}, Total Loss = {L:.2f}")

print(f"")
print(f"✅ Training complete!")
print(f"   Final w = {w:.4f}")

In [ ]:
#@title ▶️ Run this cell to visualize the learning process (click to expand code)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(w_history, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Weight (w)')
axes[0].set_title('Weight Convergence')
axes[0].grid(True, alpha=0.3)

axes[1].plot(loss_history, 'r-', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Total Loss (absolute)')
axes[1].set_title('Loss Decreasing Over Time')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("")
print("💡 What do you notice?")
print("   → With Robbins-Monro, w converges SMOOTHLY!")
print("   → No oscillation like with fixed steps!")
print("   → The loss decreases steadily.")

In [ ]:
#@title ▶️ Run this cell to visualize model fit (click to expand code)
plt.figure(figsize=(10, 5))

plt.scatter(X_train, Y_train, alpha=0.5, label='Actual Data', color='blue')

x_line = np.linspace(min(X_train), max(X_train), 100)
y_line = w * x_line
plt.plot(x_line, y_line, 'r-', linewidth=3, 
         label=f'Our Model: y = {w:.2f} × x')

plt.xlabel('Area (sq ft)')
plt.ylabel('Price ($)')
plt.title('How Well Does Our Model Fit?')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("")
print("💡 The red line is our learned model!")
print(f"   Price ≈ ${w:.2f} × Area")

---

# 🌟 Stage 4: Multiple Features (Area + Age)

## The Problem with One Feature

So far, we only used **size** to predict price. But houses also have **age**!

An old house might be cheaper than a new house of the same size.

## The New Model

Our new prediction formula:

$$\hat{y} = w_1 \times x_1 + w_2 \times x_2 + b$$

Where:
- $x_1$ = area of the house (feature 1)
- $x_2$ = age of the house (feature 2)
- $w_1$ = weight for area (how much does each sq ft add to price?)
- $w_2$ = weight for age (how much does each year affect price?)
- $b$ = bias (base price)
- $\hat{y}$ = predicted price

**Example**: If $w_1 = 200$, $w_2 = -500$, $b = 50000$:
- A 1500 sq ft, 10 year old house = $200 \times 1500 + (-500) \times 10 + 50000 = \$345,000$

## Interactive Visualization

Before we code this, let's build an interactive heatmap to explore how $w_1$ and $w_2$ affect predictions!

In [ ]:
#@title ▶️ Run this cell for interactive visualization (click to expand code)
sample_size = 100
sample_areas = areas[:sample_size]
sample_ages = ages[:sample_size]
sample_prices = prices[:sample_size]

def interactive_heatmap(w1, w2, b):
    area_range = np.linspace(sample_areas.min(), sample_areas.max(), 50)
    age_range = np.linspace(sample_ages.min(), sample_ages.max(), 50)
    area_grid, age_grid = np.meshgrid(area_range, age_range)
    price_grid = w1 * area_grid + w2 * age_grid + b
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    im = axes[0].imshow(price_grid, 
                        extent=[area_range.min(), area_range.max(), 
                                age_range.max(), age_range.min()],
                        aspect='auto', 
                        cmap='RdYlGn_r')
    
    axes[0].scatter(sample_areas, sample_ages, c=sample_prices, 
                    cmap='RdYlGn_r', edgecolors='black', s=50, alpha=0.7)
    
    axes[0].set_xlabel('Area (sq ft)')
    axes[0].set_ylabel('Age (years)')
    axes[0].set_title(f'Predicted Prices\nPrice = {w1:.1f}×Area + {w2:.1f}×Age + {b:.0f}')
    plt.colorbar(im, ax=axes[0], label='Price ($)')
    
    predictions = w1 * sample_areas + w2 * sample_ages + b
    
    axes[1].scatter(sample_prices, predictions, alpha=0.5)
    
    min_val = min(sample_prices.min(), predictions.min())
    max_val = max(sample_prices.max(), predictions.max())
    axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', 
                 linewidth=2, label='Perfect Prediction')
    
    axes[1].set_xlabel('Actual Price ($)')
    axes[1].set_ylabel('Predicted Price ($)')
    axes[1].set_title('Actual vs Predicted Prices')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    errors = np.abs(sample_prices - predictions)
    avg_error = np.mean(errors)
    print(f"📊 Average Prediction Error: ${avg_error:.2f}")

print("🎛️ Interactive Heatmap Visualization")
print("")
print("Adjust the sliders to see how w1, w2, and b affect predictions!")
print("")

interact(interactive_heatmap,
         w1=FloatSlider(min=0, max=500, step=10, value=200, 
                        description='w1 (area):'),
         w2=FloatSlider(min=-5000, max=1000, step=100, value=0, 
                        description='w2 (age):'),
         b=FloatSlider(min=-100000, max=200000, step=10000, value=0, 
                       description='b (bias):'));

### 📝 Task 4.1: Implement Multi-Feature Prediction

**Your task**: Create a function that predicts price using BOTH area AND age.

**Formula**: 
$$\hat{y} = w_1 \times x_1 + w_2 \times x_2 + b$$

In [ ]:
# ============================================
# TASK 4.1: YOUR CODE HERE
# ============================================

def predict_multi(x1, x2, w1, w2, b):
    """
    Predict house price using TWO features: area and age.
    
    What this function does:
        Calculates: price = w1×area + w2×age + b
    
    Parameters (inputs):
        x1: Area of the house (in square feet)
        
        x2: Age of the house (in years)
        
        w1: Weight for area (how much does each sq ft contribute?)
        
        w2: Weight for age (how much does each year affect price?)
        
        b:  Bias (base price before considering area and age)
    
    Returns (output):
        The predicted price
    """
    
    # TODO: Calculate and return the prediction
    
    pass

In [ ]:
#@title ▶️ Run this cell to test your predict_multi function (click to expand code)
test_pred = predict_multi(1500, 10, 200, -500, 50000)

print("🧪 Testing predict_multi:")
print(f"   House: 1500 sq ft, 10 years old")
print(f"   Weights: w1=200, w2=-500, b=50000")
print(f"   ")
print(f"   Calculation:")
print(f"     Area contribution: 200 × 1500 = ${200*1500}")
print(f"     Age contribution:  -500 × 10 = ${-500*10}")
print(f"     Base price (b):              = $50000")
print(f"     ─────────────────────────────────────")
print(f"     Total predicted:             = ${test_pred}")

if test_pred == 345000:
    print("   ✅ Correct!")
else:
    print("   ❌ Check your calculation...")

### 📝 Task 4.2: Implement Multi-Feature Loss

**Your task**: Create a function that computes total loss for multiple houses using multiple features.

In [ ]:
# ============================================
# TASK 4.2: YOUR CODE HERE
# ============================================

def compute_loss_multi(X1_list, X2_list, Y_list, w1, w2, b):
    """
    Compute total loss for multiple houses using two features.
    
    What this function does:
        1. Loops through all houses
        2. For each house, predicts the price using area AND age
        3. Calculates the error (actual - predicted)
        4. Adds all errors together
    
    Parameters (inputs):
        X1_list: List of house areas
        X2_list: List of house ages
        Y_list:  List of actual prices
        w1, w2, b: Model parameters
    
    Returns:
        Total loss (sum of all individual losses)
    """
    
    total_loss = 0
    
    # TODO: Loop through all houses and sum up the losses
    
    pass  # Remove this and add your code
    
    return total_loss

### 📝 Task 4.3: Implement Robbins-Monro for Multiple Parameters

Now we need to update **THREE parameters**: $w_1$, $w_2$, and $b$.

## 🔑 The Key Insight: Each Parameter Gets Its Own Gradient!

When we have multiple parameters, we need to ask: **"How much did THIS parameter contribute to the error?"**

Think about it this way:
- If the **area** is large, then $w_1$ has a BIG impact on the prediction
- If the **age** is large, then $w_2$ has a BIG impact on the prediction
- The **bias** $b$ always contributes the same amount (just +b)

## The Gradient Update Rules

For each house, the error is: $\text{error} = \text{actual} - \text{predicted}$

The update rules are:
- $w_1 = w_1 + \alpha \times \sum(\text{error} \times \text{area})$ ← Weighted by area!
- $w_2 = w_2 + \alpha \times \sum(\text{error} \times \text{age})$ ← Weighted by age!
- $b = b + \alpha \times \sum(\text{error})$ ← Just the sum of errors

## 💡 Why Multiply by the Feature?

**Example**: Suppose we predict $10,000 too low.
- House A: 500 sq ft → Area only contributed a little to the error
- House B: 3000 sq ft → Area contributed A LOT to the error

So we should increase $w_1$ MORE for House B than House A!

That's why we multiply the error by the feature value.

In [ ]:
# ============================================
# TASK 4.3: YOUR CODE HERE
# ============================================

# --- Setup: Use first 100 houses ---
X1_train = areas[:100]   # Areas
X2_train = ages[:100]    # Ages
Y_train = prices[:100]   # Prices

print(f"🏘️ Training on {len(X1_train)} houses with 2 features (area & age)")
print("")

# --- Initialize parameters ---
w1 = 50       # Weight for area (start with a guess)
w2 = 0        # Weight for age (start at 0)
b = 0         # Bias (start at 0)
alpha = 0.0000000001  # Very small learning rate
epochs = 100  # Number of training rounds

# --- Track history ---
w1_history = [w1]
w2_history = [w2]
b_history = [b]
loss_history = []

# --- Training Loop ---
for epoch in range(epochs):
    
    # ============================================
    # TODO: Implement the gradient descent update
    # ============================================
    # 1. Initialize gradient variables to 0:
    #    grad_w1 = 0, grad_w2 = 0, grad_b = 0
    # 
    # 2. Loop through each house (i from 0 to len(X1_train)):
    #    - Get house data: area, age, actual_price
    #    - Make prediction using predict_multi()
    #    - Calculate error = actual - predicted
    #    - Update gradients:
    #      grad_w1 += error * area
    #      grad_w2 += error * age  
    #      grad_b += error
    # 
    # 3. Update parameters:
    #    w1 = w1 + alpha * grad_w1
    #    w2 = w2 + alpha * grad_w2
    #    b = b + alpha * grad_b
    #
    # ============================================
    
    pass  # Remove this and add your code
    
    # Calculate total loss for tracking (sum of absolute errors)
    total_loss = sum(abs(Y_train[i] - predict_multi(X1_train[i], X2_train[i], w1, w2, b)) 
                     for i in range(len(X1_train)))
    
    # Track progress
    w1_history.append(w1)
    w2_history.append(w2)
    b_history.append(b)
    loss_history.append(total_loss)
    
    # Print every 20 epochs
    if epoch % 20 == 0:
        print(f"Epoch {epoch}: w1={w1:.4f}, w2={w2:.4f}, b={b:.2f}, Loss={total_loss:.0f}")

print("")
print("✅ Training complete!")
print(f"")
print(f"📊 Final parameters:")
print(f"   w1 (area weight) = {w1:.4f}")
print(f"   w2 (age weight)  = {w2:.4f}")
print(f"   b  (bias)        = {b:.2f}")

In [ ]:
#@title ▶️ Run this cell to visualize parameter convergence (click to expand code)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(w1_history, 'b-', linewidth=2)
axes[0, 0].set_title('w1 (Area Weight)')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Value')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(w2_history, 'g-', linewidth=2)
axes[0, 1].set_title('w2 (Age Weight)')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Value')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(b_history, 'm-', linewidth=2)
axes[1, 0].set_title('b (Bias)')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Value')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(loss_history, 'r-', linewidth=2)
axes[1, 1].set_title('Total Loss (Absolute)')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 All three parameters are being updated together!")
print("   The same algorithm scales to ANY number of features!")

In [ ]:
#@title ▶️ Run this cell to evaluate the model (click to expand code)
print("📊 Model Evaluation")
print("=" * 50)
print("")

predictions = []
for i in range(len(X1_train)):
    pred = predict_multi(X1_train[i], X2_train[i], w1, w2, b)
    predictions.append(pred)

predictions = np.array(predictions)

errors = np.abs(Y_train - predictions)
avg_error = np.mean(errors)
avg_price = np.mean(Y_train)

print(f"📈 Final Model:")
print(f"   Price = {w1:.2f} × Area + {w2:.2f} × Age + {b:.2f}")
print("")
print(f"📉 Error Analysis:")
print(f"   Average Prediction Error: ${avg_error:.2f}")
print(f"   Average House Price:      ${avg_price:.2f}")
print(f"   Error as % of Price:      {100 * avg_error / avg_price:.1f}%")

In [ ]:
#@title ▶️ Run this cell to see final visualization (click to expand code)
plt.figure(figsize=(10, 6))

plt.scatter(Y_train, predictions, alpha=0.5, color='blue', label='Predictions')

min_val = min(Y_train.min(), predictions.min())
max_val = max(Y_train.max(), predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, 
         label='Perfect Prediction')

plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Multi-Feature Model: Actual vs Predicted Prices')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("")
print("💡 Points closer to the red line = better predictions!")
print("   Our model has learned to predict house prices using area and age!")

---

# 🎓 Summary: What You Learned

Congratulations! You've implemented the **Robbins-Monro algorithm** from scratch!

## Key Concepts

| Stage | What You Learned |
|-------|------------------|
| **Stage 2** | Single house prediction with fixed steps (causes oscillation) |
| **Stage 3** | Batch learning with Robbins-Monro (smooth convergence!) |
| **Stage 4** | Multiple features (same algorithm works for any number!) |

## The Algorithm

The Robbins-Monro update rule:

$$w_{new} = w_{old} + \alpha \times (\text{actual} - \text{predicted})$$

**Why it works**:
- Large errors → Large corrections
- Small errors → Small corrections
- Eventually converges to optimal values!

## Key Takeaways

1. ✅ **Proportional updates** are better than fixed steps
2. ✅ The **learning rate (α)** controls speed vs stability
3. ✅ The algorithm works for **any number of features**
4. ✅ This is the foundation of **machine learning**!

---

## 🚀 What's Next?

- Try different learning rates and see what happens
- Add more features (bedrooms, distance to center)
- Compare with scikit-learn's LinearRegression
- Apply to your own datasets!

In [ ]:
#@title ▶️ Run this cell to celebrate! 🎉 (click to expand code)
print("🎉 " + "=" * 50 + " 🎉")
print("")
print("   CONGRATULATIONS!")
print("")
print("   You've successfully implemented the Robbins-Monro")
print("   stochastic approximation algorithm!")
print("")
print("   This is the foundation of how machines learn.")
print("")
print("🎉 " + "=" * 50 + " 🎉")